# Resolving Conflicts by Encoding the Precedences into the Grammar

The notebook [01-Conflicts.ipynb](01-Conflicts.ipynb) has shown that the following grammar is *ambiguous*,
because it does not specify the precedences of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "-" expr
         | expr "*" expr
         | expr "/" expr
         | expr "^" expr
         | "(" expr ")"
         | NUMBER
```

The parser generators of the `yacc` family, among them `Ply`, resolve the resulting *shift/reduce conflicts*
with *operator precedence declarations* of the form
```
    precedence = ( ('left' , '+', '-'),   # precedence 1
                   ('left' , '*', '/'),   # precedence 2
                   ('right', '^')         # precedence 3
                 )
```

<b style="color:red">`Lark` does not support operator precedence declarations.</b>
There is no way to attach a precedence or an associativity to a terminal, and there is no way to influence
the resolution of an individual shift/reduce conflict: `Lark` resolves *every* shift/reduce conflict in
favour of shifting.

Therefore, the precedences and associativities have to be encoded into the grammar itself.

## Encoding the Precedences

The recipe is the one that we have already used in the chapter on context-free languages:  we introduce one
syntactical variable $v_n$ for every precedence level $n$ and define the variable of a given precedence level
in terms of the variable of the next higher precedence level.  Let $o$ be an operator of precedence level $n$.

* If $o$ is **left associative**, the corresponding rule is **left recursive**:
  $$ v_n \rightarrow v_n \; o \; v_{n+1} $$
* If $o$ is **right associative**, the rule is **right recursive**:
  $$ v_n \rightarrow v_{n+1} \; o \; v_n $$
* If $o$ is **non-associative**, the rule uses neither recursion:
  $$ v_n \rightarrow v_{n+1} \; o \; v_{n+1} $$

For the grammar above we need three precedence levels plus one variable for the expressions that bind most
tightly:

| variable | precedence | operators | associativity |
|:---------|:-----------|:----------|:--------------|
| `expr`   | 1          | `+`, `-`  | left          |
| `prod`   | 2          | `*`, `/`  | left          |
| `fact`   | 3          | `^`       | right         |
| `atom`   | &mdash;    | &mdash;   | &mdash;       |

Note how the associativities show up in the grammar below:  the rules for `expr` and `prod` are *left*
recursive, while the rule `fact : atom "^" fact` is *right* recursive.

Two features of `Lark` are used to keep the parse tree readable.

1. If the name of a syntactical variable is prefixed with a question mark, as in `?expr`, then a node of this
   variable is removed from the tree whenever it has exactly one child.  Without this, every number would be
   buried under a chain `expr` &rarr; `prod` &rarr; `fact` &rarr; `atom` that carries no information.
2. An alternative can be given an *alias* using the operator `->`.  The alias becomes the name of the node
   that is created and, later on, the name of the method of our `Transformer`.

An alternative that has an alias is *never* inlined, even if it has a single child.  This is the reason why
the alternative `NUMBER -> number` survives although `?atom` starts with a question mark.

Neither of these features has any influence on the parse table:  they can neither introduce nor remove a
conflict.

In [ ]:
grammar = r"""
    ?expr : expr "+" prod   -> add
          | expr "-" prod   -> sub
          | prod

    ?prod : prod "*" fact   -> mul
          | prod "/" fact   -> div
          | fact

    ?fact : atom "^" fact   -> power
          | atom

    ?atom : "(" expr ")"
          | NUMBER          -> number

    NUMBER : /0|[1-9][0-9]*/

    %import common.WS
    %ignore WS
"""

## Specification of the Parser

We create the parser with `strict=True`.  In this mode, `Lark` raises an exception of class `GrammarError`
as soon as it finds a shift/reduce conflict, instead of silently resolving it in favour of shifting.  It is
a good idea to develop a grammar this way, for then a conflict cannot be overlooked.

Since no exception is raised below, our grammar is an `LALR(1)` grammar.

<b style="color:red">Note:</b> `strict=True` additionally checks the terminals for collisions and this check
needs the package `interegular`, which is not installed together with `Lark` by default.  If the cell below
raises a `LexError`, install it via
```
    pip install interegular
```

In [ ]:
from lark import Lark, Transformer, GrammarError

parser = Lark(grammar, start='expr', parser='lalr', strict=True)

## Inspecting the LALR States

`Ply` writes the LALR states into the file `parser.out`.  `Lark` has no such file, but if the parser is
created with `debug=True`, the parse table is kept in a form where the states are still the *sets of marked
rules* that we have discussed in the lecture.  The function `dump_states` is the same one that we have used
in [01-Conflicts.ipynb](01-Conflicts.ipynb).

Since the states are sets, iterating over them is not reproducible from one run to the next.  Therefore, the
states are sorted before they are numbered and the start state is put first, so that the numbering agrees
with the convention used by `Ply`.

In [ ]:
from lark.parsers.lalr_analysis import Shift

def dump_states(lark_parser):
    table  = lark_parser.parser.parser.parser.parse_table
    key    = lambda state: sorted(str(rule) for rule in state)
    start  = next(iter(table.start_states.values()))
    rest   = sorted((s for s in table.states if s != start), key=key)
    order  = [start] + rest
    number = { state: i for i, state in enumerate(order) }
    for state in order:
        print(f'state {number[state]}')
        for marked_rule in sorted(state, key=str):
            print(f'    {marked_rule}')
        print()
        for token, (action, arg) in sorted(table.states[state].items()):
            if action is Shift:
                print(f'    {token:<8} shift and go to state {number[arg]}')
            else:
                print(f'    {token:<8} reduce using rule {arg}')
        print()

In order to look at the states we have to build the parser a second time, now with `debug=True`.
`Lark` marks the position of the parser inside a rule with the character `*` instead of the bullet `•` that
we use in the lecture notes, and the end of the input is called `$END`.

Note that there is no state that offers both a `shift` and a `reduce` action for the same token:  all
conflicts are gone.

In [ ]:
debug_parser = Lark(grammar, start='expr', parser='lalr', debug=True)

In [ ]:
dump_states(debug_parser)

## Building an Abstract Syntax Tree

The parse tree returned by `Lark` is an object of class `Tree`.  In order to reuse the function `tuple2dot`,
we convert this tree into a nested tuple.  This is done with a `Transformer`:  for every alias occurring in
the grammar, the transformer provides a method of the same name.  This method receives the list of the
children that have already been transformed and returns the value that replaces the node.

The alternatives without an alias, that is the chain rules and the rule for parenthesized expressions, need
no method:  they have been inlined by the question mark and therefore never reach the transformer.

In [ ]:
%run ../AST2Dot.ipynb

In [ ]:
class ASTBuilder(Transformer):
    def add(self, children):
        lhs, rhs = children
        return ('+', lhs, rhs)

    def sub(self, children):
        lhs, rhs = children
        return ('-', lhs, rhs)

    def mul(self, children):
        lhs, rhs = children
        return ('*', lhs, rhs)

    def div(self, children):
        lhs, rhs = children
        return ('/', lhs, rhs)

    def power(self, children):
        lhs, rhs = children
        return ('^', lhs, rhs)

    def number(self, children):
        return int(children[0])

The function `test(s)` takes a string `s` as its argument and tries to parse this string.  If all goes well,
an abstract syntax tree is returned and displayed.  If the string can't be parsed, `Lark` raises an exception
of class `UnexpectedInput`.  The message of this exception already contains the line, the column and the
tokens that would have been acceptable, so there is no need for a function like `p_error`.

In [ ]:
from lark.exceptions import UnexpectedInput

def test(s):
    try:
        parse_tree = parser.parse(s)
    except UnexpectedInput as e:
        print(f'Syntax error:\n{e}')
        return None
    t = ASTBuilder().transform(parse_tree)
    d = tuple2dot(t)
    display(d)
    return t

In [ ]:
test('2^3^4*5+6-7/8^9')

In [ ]:
test('1+2*3^4')

In [ ]:
test('1 * 2 + (3^4)^5')

The next two examples check the associativities:  the additive operators group to the left, while the
exponentiation operator groups to the right.

In [ ]:
test('1-2-3')

In [ ]:
test('2^3^4')

Finally, an example of a string that cannot be parsed.

In [ ]:
test('1 + * 2')